In [2]:
import os
import numpy as np
import pandas as pd
from data_utils import preprocessing
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, GroupKFold, KFold
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

import warnings
warnings.filterwarnings("ignore")

In [3]:
data_path = '../../input/commonlit-evaluate-student-summaries'

prompts_df = pd.read_csv(f'{data_path}/prompts_train.csv')
summary_df = pd.read_csv(f'{data_path}/summaries_train.csv')
print(prompts_df.shape, summary_df.shape)
display(prompts_df.head(2))
display(summary_df.head(2))

df = summary_df.merge(prompts_df, on="prompt_id")
print(df.shape)
display(df.head(2))

(4, 4) (7165, 5)


,prompt_id,prompt_question,prompt_title,prompt_text
0,39c16e,Summarize at least 3 elements of an ideal trag...,On Tragedy,Chapter 13 \r\nAs the sequel to what has alrea...
1,3b9047,"In complete sentences, summarize the structure...",Egyptian Social Structure,Egyptian society was structured like a pyramid...


,student_id,prompt_id,text,content,wording
0,000e8c3c7ddb,814d6b,The third wave was an experimentto see how peo...,0.205683,0.380538
1,0020ae56ffbf,ebad26,They would rub it up with soda to make the sme...,-0.548304,0.506755


(7165, 8)


,student_id,prompt_id,text,content,wording,prompt_question,prompt_title,prompt_text
0,000e8c3c7ddb,814d6b,The third wave was an experimentto see how peo...,0.205683,0.380538,Summarize how the Third Wave developed over su...,The Third Wave,Background \r\nThe Third Wave experiment took ...
1,0070c9e7af47,814d6b,The Third Wave developed rapidly because the ...,3.272894,3.219757,Summarize how the Third Wave developed over su...,The Third Wave,Background \r\nThe Third Wave experiment took ...


In [4]:
df['full_text'] = df["prompt_title"] + " " + df["prompt_text"]+ " " + df["prompt_question"]+ " " + df["text"]
df['full_text_processed'] =  df['full_text'].apply(lambda x: preprocessing(x, "p1"))
df['full_text_len'] = df['full_text_processed'].apply(lambda x: len(x.split()))
df['unique_word_len'] = df['full_text_processed'].apply(lambda x: len(set(x.split())))

In [5]:
# c1
n_folds=4
seed=42

nFolds = GroupKFold(n_splits=n_folds)
for n, (train_index, val_index) in enumerate(nFolds.split(X=df, groups=df["prompt_id"])):
    df.loc[val_index, 'fold'] = int(n)
df['fold'] = df['fold'].astype(int)

cols = ["content", "wording", "full_text_len", "unique_word_len"]

display(df.groupby(['fold'])[cols].agg(['mean', 'std']))
display(pd.crosstab(df["fold"], df["prompt_id"]))
display(pd.crosstab(df["fold"], df["content"]))
display(pd.crosstab(df["fold"], df["wording"]))
display(pd.crosstab(df["fold"], df["full_text_len"]))
display(pd.crosstab(df["fold"], df["unique_word_len"]))

content             wording           full_text_len             \
          mean       std      mean       std          mean        std   
fold                                                                    
0    -0.095457  0.969773 -0.140749  1.055695    685.832766  44.525331   
1     0.049579  1.106129 -0.068542  0.952708    679.531608  60.476981   
2    -0.087906  0.990271 -0.299023  0.930270   1071.450902  55.690463   
3     0.150306  1.124158  0.518733  1.107806    696.873980  46.722173   

     unique_word_len             
                mean        std  
fold                             
0         329.057365  13.516333  
1         355.125436  16.328829  
2         450.613226  11.679844  
3         330.236627  13.508995

prompt_id,39c16e,3b9047,814d6b,ebad26
fold,,,,
0,2057,0,0,0
1,0,2009,0,0
2,0,0,0,1996
3,0,0,1103,0


content,-1.729859,-1.670219,-1.638511,-1.578871,-1.547163,-1.519231,-1.515225,-1.499528,-1.487523,-1.483517,...,3.496740,3.502996,3.503226,3.626282,3.679436,3.711374,3.802722,3.834430,3.894070,3.900326
fold,,,,,,,,,,,,,,,,,,,,,
0,2,0,4,2,89,2,0,0,11,1,...,1,1,2,1,0,0,1,1,0,1
1,1,1,6,0,140,2,0,1,12,2,...,0,0,0,0,0,0,1,0,1,0
2,0,0,4,4,158,1,0,0,16,0,...,0,0,1,0,0,0,0,0,0,0
3,1,0,7,0,39,2,1,0,2,0,...,0,0,0,0,1,2,0,0,0,0


wording,-1.962614,-1.795491,-1.751663,-1.712410,-1.707835,-1.672196,-1.668582,-1.629329,-1.628368,-1.592728,...,3.638126,3.732697,3.739233,3.761422,3.775564,3.897941,4.064103,4.187398,4.231226,4.310693
fold,,,,,,,,,,,,,,,,,,,,,
0,1,11,0,0,1,10,0,3,5,1,...,2,1,0,2,1,1,0,1,0,0
1,0,37,0,1,1,1,0,2,2,0,...,0,0,0,0,0,0,0,0,1,0
2,0,40,1,1,1,4,1,4,4,0,...,1,0,0,0,0,0,0,0,0,0
3,0,5,0,0,0,0,0,1,1,0,...,0,0,1,1,0,0,2,0,0,1


full_text_len,620,621,622,623,624,625,626,627,628,629,...,1341,1351,1352,1355,1383,1386,1397,1423,1430,1501
fold,,,,,,,,,,,,,,,,,,,,,
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,10,31,29,37,38,38,25,41,34,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,2,1,1,1,1,1,1,1,1,1
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


unique_word_len,309,310,312,313,314,315,316,317,318,319,...,502,504,505,508,511,512,526,527,530,550
fold,,,,,,,,,,,,,,,,,,,,,
0,0,0,18,19,21,46,55,70,67,84,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,0,0,0,0,0,0,0,0,0,0,...,2,1,1,1,1,1,1,1,2,0
3,7,2,6,10,11,12,23,22,53,44,...,0,0,0,0,0,0,0,0,0,0


In [6]:
# c2
n_folds=4
seed=42


nFolds = MultilabelStratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
for n, (train_index, val_index) in enumerate(nFolds.split(X=df, y=df[cols].values)):
    df.loc[val_index, 'fold'] = int(n)
df['fold'] = df['fold'].astype(int)

cols = ["content", "wording", "full_text_len", "unique_word_len"]

display(df.groupby(['fold'])[cols].agg(['mean', 'std']))
display(pd.crosstab(df["fold"], df["prompt_id"]))
display(pd.crosstab(df["fold"], df["content"]))
display(pd.crosstab(df["fold"], df["wording"]))
display(pd.crosstab(df["fold"], df["full_text_len"]))
display(pd.crosstab(df["fold"], df["unique_word_len"]))

content             wording           full_text_len              \
          mean       std      mean       std          mean         std   
fold                                                                     
0    -0.007777  1.038373 -0.081509  1.028604    799.733110  185.677702   
1    -0.020464  1.048148 -0.058223  1.052726    793.939698  180.582585   
2    -0.037222  1.041890 -0.069576  1.028281    786.406250  176.640519   
3     0.006063  1.046225 -0.042976  1.034873    792.683975  180.513311   

     unique_word_len             
                mean        std  
fold                             
0         372.262423  54.201984  
1         370.875489  52.584697  
2         368.212612  51.831654  
3         370.293691  52.616084

prompt_id,39c16e,3b9047,814d6b,ebad26
fold,,,,
0,507,494,267,523
1,508,503,273,507
2,537,505,277,473
3,505,507,286,493


content,-1.729859,-1.670219,-1.638511,-1.578871,-1.547163,-1.519231,-1.515225,-1.499528,-1.487523,-1.483517,...,3.496740,3.502996,3.503226,3.626282,3.679436,3.711374,3.802722,3.834430,3.894070,3.900326
fold,,,,,,,,,,,,,,,,,,,,,
0,1,0,6,2,102,3,1,0,8,3,...,0,0,1,0,0,0,0,1,0,0
1,0,0,7,2,108,2,0,0,13,0,...,1,0,2,0,1,2,1,0,0,0
2,2,1,3,1,116,1,0,0,12,0,...,0,1,0,0,0,0,0,0,1,0
3,1,0,5,1,100,1,0,1,8,0,...,0,0,0,1,0,0,1,0,0,1


wording,-1.962614,-1.795491,-1.751663,-1.712410,-1.707835,-1.672196,-1.668582,-1.629329,-1.628368,-1.592728,...,3.638126,3.732697,3.739233,3.761422,3.775564,3.897941,4.064103,4.187398,4.231226,4.310693
fold,,,,,,,,,,,,,,,,,,,,,
0,0,27,0,2,0,2,0,1,2,1,...,1,0,1,1,0,0,0,1,0,0
1,1,20,1,0,1,5,0,6,5,0,...,2,1,0,1,0,0,2,0,1,1
2,0,20,0,0,0,3,1,1,5,0,...,0,0,0,0,0,1,0,0,0,0
3,0,26,0,0,2,5,0,2,0,0,...,0,0,0,1,1,0,0,0,0,0


full_text_len,620,621,622,623,624,625,626,627,628,629,...,1341,1351,1352,1355,1383,1386,1397,1423,1430,1501
fold,,,,,,,,,,,,,,,,,,,,,
0,0,2,6,7,9,8,8,6,8,9,...,0,0,0,1,0,1,1,0,0,1
1,1,2,11,7,7,9,12,5,11,3,...,1,0,1,0,0,0,0,0,1,0
2,0,3,5,7,12,13,11,8,12,14,...,0,1,0,0,0,0,0,0,0,0
3,0,3,9,8,9,8,7,6,10,8,...,1,0,0,0,1,0,0,1,0,0


unique_word_len,309,310,312,313,314,315,316,317,318,319,...,502,504,505,508,511,512,526,527,530,550
fold,,,,,,,,,,,,,,,,,,,,,
0,2,1,6,11,12,7,27,25,26,39,...,0,0,1,1,1,1,0,0,1,1
1,0,0,6,6,7,13,14,28,34,26,...,0,0,0,0,0,0,0,1,0,0
2,2,1,5,9,4,20,15,18,31,29,...,1,1,0,0,0,0,1,0,0,0
3,3,0,7,3,9,18,22,21,29,34,...,1,0,0,0,0,0,0,0,1,0


In [7]:
# c3
n_folds=4
seed=42

num_bins = int(np.floor(1+(3.3)*(np.log2(len(df)))))
df[f"content_bins"] = pd.cut(df[f"content"], bins=num_bins, labels=False)
df[f"wording_bins"] = pd.cut(df[f"wording"], bins=num_bins, labels=False)

st_cols = ["content_bins", "wording_bins", "full_text_len", "unique_word_len"]

nFolds = MultilabelStratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
for n, (train_index, val_index) in enumerate(nFolds.split(X=df, y=df[st_cols].values)):
    df.loc[val_index, 'fold'] = int(n)
df['fold'] = df['fold'].astype(int)

cols = ["content", "wording", "full_text_len", "unique_word_len"]

display(df.groupby(['fold'])[cols].agg(['mean', 'std']))
display(pd.crosstab(df["fold"], df["prompt_id"]))
display(pd.crosstab(df["fold"], df["content"]))
display(pd.crosstab(df["fold"], df["wording"]))
display(pd.crosstab(df["fold"], df["full_text_len"]))
display(pd.crosstab(df["fold"], df["unique_word_len"]))

content             wording           full_text_len              \
          mean       std      mean       std          mean         std   
fold                                                                     
0    -0.006621  1.032470 -0.066918  1.023319    791.129537  179.678599   
1    -0.030319  1.049132 -0.060028  1.058857    797.293132  182.012405   
2    -0.016023  1.034921 -0.052799  1.020562    792.734375  180.327166   
3    -0.006448  1.058240 -0.072548  1.041764    791.602457  181.679866   

     unique_word_len             
                mean        std  
fold                             
0         370.040201  52.352293  
1         371.151870  53.564646  
2         370.261719  52.486707  
3         370.189280  52.923444

prompt_id,39c16e,3b9047,814d6b,ebad26
fold,,,,
0,513,524,266,488
1,516,465,290,520
2,516,503,277,496
3,512,517,270,492


content,-1.729859,-1.670219,-1.638511,-1.578871,-1.547163,-1.519231,-1.515225,-1.499528,-1.487523,-1.483517,...,3.496740,3.502996,3.503226,3.626282,3.679436,3.711374,3.802722,3.834430,3.894070,3.900326
fold,,,,,,,,,,,,,,,,,,,,,
0,1,1,4,0,107,2,1,1,6,0,...,1,0,0,1,0,0,1,0,0,0
1,0,0,7,1,120,3,0,0,16,1,...,0,0,2,0,1,0,1,1,1,0
2,2,0,5,3,98,2,0,0,11,1,...,0,1,1,0,0,2,0,0,0,0
3,1,0,5,2,101,0,0,0,8,1,...,0,0,0,0,0,0,0,0,0,1


wording,-1.962614,-1.795491,-1.751663,-1.712410,-1.707835,-1.672196,-1.668582,-1.629329,-1.628368,-1.592728,...,3.638126,3.732697,3.739233,3.761422,3.775564,3.897941,4.064103,4.187398,4.231226,4.310693
fold,,,,,,,,,,,,,,,,,,,,,
0,1,28,0,0,0,4,0,3,2,1,...,0,1,1,0,1,0,0,0,0,0
1,0,24,0,1,0,3,1,4,3,0,...,2,0,0,0,0,0,0,1,1,0
2,0,25,0,1,1,3,0,1,3,0,...,1,0,0,2,0,1,2,0,0,0
3,0,16,1,0,2,5,0,2,4,0,...,0,0,0,1,0,0,0,0,0,1


full_text_len,620,621,622,623,624,625,626,627,628,629,...,1341,1351,1352,1355,1383,1386,1397,1423,1430,1501
fold,,,,,,,,,,,,,,,,,,,,,
0,0,2,9,11,6,8,8,8,6,8,...,0,0,1,0,0,0,0,0,0,0
1,0,3,10,7,5,14,7,6,9,11,...,1,1,0,0,0,1,1,0,0,1
2,0,4,5,5,14,6,10,7,13,10,...,1,0,0,0,0,0,0,0,0,0
3,1,1,7,6,12,10,13,4,13,5,...,0,0,0,1,1,0,0,1,1,0


unique_word_len,309,310,312,313,314,315,316,317,318,319,...,502,504,505,508,511,512,526,527,530,550
fold,,,,,,,,,,,,,,,,,,,,,
0,4,0,12,7,11,12,14,22,26,29,...,1,0,0,1,0,0,0,0,0,0
1,0,1,6,9,10,11,23,31,24,31,...,1,0,1,0,0,0,0,0,1,1
2,1,0,5,6,6,13,22,16,40,32,...,0,1,0,0,0,1,1,0,0,0
3,2,1,1,7,5,22,19,23,30,36,...,0,0,0,0,1,0,0,1,1,0
